# Implementing Advantage-Actor Critic (A2C) - 2 pts

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel. 

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscal, take max between frames, skip frames, stack them together, prepares for PyTorch and normalizes to [0, 1]) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function.

In [1]:
!pip install gymnasium==1.0.0
!pip install ale-py==0.10.2
!pip install opencv-python
!pip install "gymnasium[other]"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import numpy as np
from atari_wrappers import nature_dqn_env
import gymnasium as gym
from atari_wrappers import TensorboardSummaries

nenvs = 64    # change this if you have more than 8 CPU ;)
env = gym.vector.AsyncVectorEnv([lambda: nature_dqn_env("SpaceInvadersNoFrameskip-v4") for _ in range(nenvs)])
env = TensorboardSummaries(env, "spaceinvaders")


n_actions = env.single_action_space.n
obs, info = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]


Next, we will need to implement a model that predicts logits of policy distribution and critic value. Use shared backbone. You may use same architecture as in DQN task with one modification: instead of having a single output layer, it must have two output layers taking as input the output of the last hidden layer (one for actor, one for critic). 

Still it may be very helpful to make more changes:
* use orthogonal initialization with gain $\sqrt{2}$ and initialize biases with zeros;
* use more filters (e.g. 32-64-64 instead of 16-32-64);
* use two-layer heads for actor and critic or add a linear layer into backbone;

**Danger:** do not divide on 255, input is already normalized to [0, 1] in our wrappers!

In [3]:
import torch
from torch import nn

class ActorCritic(nn.Module):
    def __init__(self, n_actions: int, inp_size: int = 64 * 7 * 7, hidden_size: int = 512):
        super().__init__()
        self.conv_backbone = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.GELU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.GELU(),
            nn.Flatten(),
        )
        self.actor_head = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, n_actions),
        )
        self.critic_head = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 1),
        )
    
    @property
    def device(self):
        return next(self.parameters()).device
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.conv_backbone(x)
        actor_logits = self.actor_head(x)
        critic_value = self.critic_head(x).squeeze(-1)
        return actor_logits, critic_value



You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a **dictionary** of all the arrays that are needed to interact with an environment and train the model.

**Important**: "actions" will be sent to environment, they must be numpy array or list, not PyTorch tensor.

Note: you can add more keys, e.g. it can be convenient to compute entropy right here.

In [4]:
from torch.distributions import Categorical

class Policy:
    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def act(self, inputs):
        '''
        input:
            inputs - numpy array, (batch_size x channels x width x height)
        output: dict containing keys ['actions', 'logits', 'log_probs', 'values']:
            'actions' - selected actions, numpy, (batch_size)
            'logits' - actions logits, tensor, (batch_size x num_actions)
            'log_probs' - log probs of selected actions, tensor, (batch_size)
            'values' - critic estimations, tensor, (batch_size)
        '''
        inputs_pt = torch.from_numpy(inputs).float().to(self.model.device)
        logits, value = self.model(inputs_pt)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        
        return {
            "actions": action.numpy(force=True),
            "logits": logits,
            "log_probs": log_prob,
            "values": value,
        }

Next we will pass the environment and policy to a runner that collects rollouts from the environment. 
The class is already implemented for you.

In [5]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys 

* 'observations' 
* 'rewards' 
* 'dones'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment of specified length $T$ &mdash; the size of partial trajectory, or rollout length. Let's have a look at how it works.

In [6]:
model = ActorCritic(n_actions)
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=5)

In [7]:
# generates new rollout
trajectory = runner.get_next()

In [8]:
# what is inside
print(trajectory.keys())

dict_keys(['actions', 'logits', 'log_probs', 'values', 'observations', 'rewards', 'dones'])


In [9]:
# Sanity checks
assert 'logits' in trajectory, "Not found: policy didn't provide logits"
assert 'log_probs' in trajectory, "Not found: policy didn't provide log_probs of selected actions"
assert 'values' in trajectory, "Not found: policy didn't provide critic estimations"
assert trajectory['logits'][0].shape == (nenvs, n_actions), "logits wrong shape"
assert trajectory['log_probs'][0].shape == (nenvs,), "log_probs wrong shape"
assert trajectory['values'][0].shape == (nenvs,), "values wrong shape"

for key in trajectory.keys():
    assert len(trajectory[key]) == 5, \
    f"something went wrong: 5 steps should have been done, got trajectory of length {len(trajectory[key])} for '{key}'"

Now let's work with this trajectory a bit. To train the critic you will need to compute the value targets. It will also be used as an estimation of $Q$ for actor training.

You should use all available rewards for value targets, so the formula for the value targets is simple:

$$
\hat v(s_t) = \sum_{t'=0}^{T - 1}\gamma^{t'}r_{t+t'} + \gamma^T \hat{v}(s_{t+T}),
$$

where $s_{t + T}$ is the latest observation of the environment.

Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected. 
Thus, we can implement and use `ComputeValueTargets` callable. 

**Do not forget** to use `trajectory['dones']` flags to check if you need to add the value targets at the next step when 
computing value targets for the current step.

**Bonus (+0.5 pts):** implement [Generalized Advantage Estimation (GAE)](https://arxiv.org/pdf/1506.02438.pdf) instead; use $\lambda \approx 0.95$ or even closer to 1 in experiment. 

In [10]:
trajectory["values"]

[tensor([0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072]),
 tensor([0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072, 0.0072,
         

In [11]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99, gae_lambda=0.95):
        self.policy = policy
        self.gamma = gamma
        self.gae_lambda = gae_lambda

    def __call__(self, trajectory, latest_observation):
        '''
        This method should modify trajectory inplace by adding 
        an item with key 'value_targets' to it
        
        input:
            trajectory - dict from runner
            latest_observation - last state, numpy, (num_envs x channels x width x height)
        '''
        policy_output = self.policy.act(latest_observation)
        last_value = policy_output['values']
        values = trajectory['values']
        rewards = trajectory['rewards']
        terms = trajectory['dones']
        advantages = []
        last_gae_lambda = torch.zeros_like(values[-1])
        for t in reversed(range(len(rewards))):
            nonterms_t = 1 - torch.tensor(terms[t], dtype=torch.float32, device=self.policy.model.device)
            delta = torch.tensor(rewards[t], dtype=torch.float32, device=self.policy.model.device) + self.gamma * nonterms_t * last_value - values[t]
            last_gae_lambda = delta + self.gamma * self.gae_lambda * nonterms_t * last_gae_lambda
            last_value = values[t]
            advantages.append(last_gae_lambda)
        advantages.reverse()
        value_targets = list(map(lambda x, y: x + y, values, advantages))
        
        trajectory['value_targets'] = value_targets
        trajectory['gae'] = advantages

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `T * nenvs`.

You need to make sure that after this transformation `"log_probs"`, `"value_targets"`, `"values"` are 1-dimensional PyTorch tensors.

In [12]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory, latest_observation):
        # Modify trajectory inplace. 
        for k, v in trajectory.items():
            if isinstance(v[0], np.ndarray):
                trajectory[k] = torch.from_numpy(np.concatenate(v))
            else:
                trajectory[k] = torch.cat(v)


Let's do more sanity checks!

In [13]:
runner = EnvRunner(env, policy, nsteps=5, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

trajectory = runner.get_next()

In [14]:
# More sanity checks
assert 'value_targets' in trajectory, "Value targets not found"
assert trajectory['log_probs'].shape == (5 * nenvs,)
assert trajectory['value_targets'].shape == (5 * nenvs,)
assert trajectory['values'].shape == (5 * nenvs,)

assert not trajectory['log_probs'].requires_grad, "Gradients are not needed."
assert not trajectory['values'].requires_grad, "Gradients are not needed."

Now is the time to implement the advantage actor critic algorithm itself. You can look into [Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and lectures ([part 1](https://www.youtube.com/watch?v=Ds1trXd6pos&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=5), [part 2](https://www.youtube.com/watch?v=EKqxumCuAAY&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=6)) by Sergey Levine.

In [15]:
from tqdm import tqdm
from collections import defaultdict
from torch.nn.utils import clip_grad_norm_


class PPO:
    def __init__(
        self,
        policy,
        optimizer,
        num_updates: int = 10000,
        value_loss_coef: float = 0.25,
        entropy_coef: float = 0.01,
        max_grad_norm: float = 0.5,
        num_epochs: int = 2,
        num_minibatches: int = 4,
        pg_eps: float = 0.2,
        vf_eps: float = 1.0,
    ):
        self.num_updates = num_updates
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.num_epochs = num_epochs
        self.num_minibatches = num_minibatches
        self.pg_eps = pg_eps
        self.vf_eps = vf_eps

    def update(self, trajectory, write):
        # compute all losses
        # do not forget to use weights for critic loss and entropy loss
        B = len(trajectory["observations"])
        pg_losses = []
        vf_losses = []
        entropy_losses = []
        clipfracs = []
        adv = trajectory["gae"]
        adv_mean = adv.mean()
        adv_std = adv.std()
        trajectory["observations"] = trajectory["observations"].to(self.policy.model.device)
        trajectory["actions"] = trajectory["actions"].to(self.policy.model.device)
        trajectory["gae_normalized"] = (adv - adv_mean) / (adv_std + 1e-8)
        mb_size = B // self.num_minibatches
        for epoch in range(self.num_epochs):
            ids = torch.randperm(B)
            for i in range(0, B, mb_size):
                mb_ids = ids[i : i + mb_size]
                obs = trajectory["observations"][mb_ids]
                actions = trajectory["actions"][mb_ids]
                old_log_probs = trajectory["log_probs"][mb_ids]
                old_values = trajectory["values"][mb_ids]
                adv = trajectory["gae_normalized"][mb_ids]
                value_targets = trajectory["value_targets"][mb_ids]
                log_probs, values = self.policy.model(obs)
                dist = Categorical(logits=log_probs)
                new_log_probs = dist.log_prob(actions)
                entropy = dist.entropy()
                ratio = torch.exp(new_log_probs - old_log_probs)
                surr1 = ratio * adv
                surr2 = torch.clamp(ratio, 1.0 - self.pg_eps, 1.0 + self.pg_eps) * adv
                clipfrac = torch.mean((torch.abs(ratio - 1.0) > self.pg_eps).float())
                pg_loss = -torch.mean(torch.min(surr1, surr2))
                values_clipped = old_values + torch.clamp(values - old_values, -self.vf_eps, self.vf_eps)
                vf_loss_unclipped = (values - value_targets) ** 2
                vf_loss_clipped = (values_clipped - value_targets) ** 2
                vf_loss = 0.5 * torch.mean(torch.max(vf_loss_unclipped, vf_loss_clipped))
                entropy_loss = -torch.mean(entropy)
                loss = pg_loss + self.entropy_coef * entropy_loss + self.value_loss_coef * vf_loss
                self.optimizer.zero_grad()
                loss.backward()
                clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
                self.optimizer.step()

                pg_losses.append(pg_loss.numpy(force=True))
                vf_losses.append(vf_loss.numpy(force=True))
                entropy_losses.append(entropy_loss.numpy(force=True))
                clipfracs.append(clipfrac.numpy(force=True))

        value_target = float(torch.mean(trajectory["value_targets"]).numpy(force=True))
        clipfrac = float(np.mean(clipfracs))

        # log all losses
        write(
            "losses",
            {
                "policy loss": float(np.mean(pg_losses)),
                "critic loss": float(np.mean(vf_losses)),
                "entropy loss": float(np.mean(entropy_losses)),
            },
        )

        # additional logs
        write("policy/clipfrac", clipfrac)
        write("critic/advantage", float(adv_mean.numpy(force=True)))
        write("critic/values", {"value targets": value_target})

        return value_target

    def train(self, runner):
        # collect trajectory using runner
        # compute loss and perform one step of gradient optimization
        # do not forget to clip gradients
        pbar = tqdm(range(self.num_updates), desc="Training")
        for _ in pbar:
            trajectory = runner.get_next()
            val_target = self.update(trajectory, runner.write)
            pbar.set_postfix({"value_target": val_target})


Now you can train your model. For optimization we suggest you use RMSProp with learning rate 7e-4 (you can also linearly decay it to 0), smoothing constant (alpha in PyTorch) equal to 0.99 and epsilon equal to 1e-5.

We recommend to train for at least 10 million environment steps across all batched environments (takes ~3 hours on a single GTX1080 with 8 CPU). It should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last episodes in each environment in the batch) of about 600. **Your goal is to reach 500**.

Notes:
* if your reward is stuck at ~200 for more than 2M steps then probably there is a bug
* if your gradient norm is >10 something probably went wrong
* make sure your `entropy loss` is negative, your `critic loss` is positive
* make sure you didn't forget `.detach` in losses where it's needed
* `actor loss` should oscillate around zero or near it; do not expect loss to decrease in RL ;)
* you can experiment with `nsteps` ("rollout length"); standard rollout length is 5 or 10. Note that this parameter influences how many algorithm iterations is required to train on 10M steps (or 40M frames --- we used frameskip in preprocessing).

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ActorCritic(n_actions).to(device)
policy = Policy(model)

nsteps = 32
runner = EnvRunner(env, policy, nsteps=nsteps, transforms=[ComputeValueTargets(policy, gamma=0.995, gae_lambda=0.99), MergeTimeBatch()])

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

ppo = PPO(policy, optimizer, num_updates=10000)

In [18]:
ppo.train(runner)

Training:  13%|█▎        | 1308/10000 [14:55<1:39:11,  1.46it/s, value_target=5]   


KeyboardInterrupt: 

In [29]:
# save your model just in case 
torch.save(model.state_dict(), "ppo.pt")    

In [19]:
env.close()

/usr/local/lib/python3.11/dist-packages/gymnasium/vector/async_vector_env.py:573: UserWarning: WARN: Calling `close` while waiting for a pending call to `step` to complete.
  logger.warn(
/usr/local/lib/python3.11/dist-packages/gymnasium/vector/async_vector_env.py:422: UserWarning: ERROR: Received the following error from Worker-36 - Shutting it down
  self._raise_if_errors(successes)
/usr/local/lib/python3.11/dist-packages/gymnasium/vector/async_vector_env.py:422: UserWarning: ERROR: Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gymnasium/vector/async_vector_env.py", line 701, in _async_worker
    command, data = pipe.recv()
                    ^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/connection.py", line 250, in recv
    buf = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/connection.py", line 430, in _recv_bytes
    buf = self._recv(4)
          ^^^^^^^^^^^^^
  File "/usr/lib/python3.11/

KeyboardInterrupt: 

## Evaluation

In [20]:
env = nature_dqn_env("SpaceInvadersNoFrameskip-v4", clip_reward=False, episodic_life=False)

In [21]:
def evaluate(env, policy, n_games=1, t_max=10000):
    '''
    Plays n_games and returns rewards
    '''
    rewards = []
    
    for _ in range(n_games):
        s, info = env.reset()
        
        R = 0
        for _ in range(t_max):
            action = policy.act(np.array([s]))["actions"][0]
            
            s, r, term, trank, _ = env.step(action)
            
            R += r
            if term or trank:
                break

        rewards.append(R)
    return np.array(rewards)

In [23]:
# evaluation will take some time!
sessions = evaluate(env, policy, n_games=30)
score = sessions.mean()
print(f"Your score: {score}")

assert score >= 500, "Needs more training?"
print("Well done!")

Your score: 574.6666666666666
Well done!


In [24]:
env.close()

## Record

In [25]:
env_monitor = nature_dqn_env("SpaceInvadersNoFrameskip-v4", monitor=True, clip_reward=False, episodic_life=False)

In [26]:
# record sessions
sessions = evaluate(env_monitor, policy, n_games=3)

In [27]:
# rewards for recorded games
sessions

array([515., 435., 600.])

In [28]:
env_monitor.close()